## לקרוא את ההיסטוריה: `git log`, `git diff`, `git status`

בסעיף הקודם למדנו **לכתוב** היסטוריה. עכשיו נלמד **לקרוא** אותה — כישור שנדרש הרבה יותר בפועל, כי לרוב אתם לא זוכרים בעצמכם מה קרה בכל קומיט, או שאתם קוראים היסטוריה שכתב מישהו אחר (או "אתם" של לפני חודש).

- **`git status`** — מה השתנה מאז הקומיט האחרון, ומה עדיין לא נשמר?
- **`git log --oneline`** — רשימת כל הקומיטים, כל אחד עם hash קצר (מזהה) והודעה.
- **`git diff <hash1> <hash2> -- <קובץ>`** — בדיוק אילו שורות השתנו בין שני קומיטים, שורה-שורה.
- **`git show <hash>:<קובץ>`** — התוכן המלא של קובץ כפי שהיה באותו קומיט.

נשתמש בהם יחד כדי לפתור תעלומה: קוד שרץ בלי שגיאה, אבל מחזיר תוצאה פיזיקלית שגויה — בדיוק כמו מאגר הבאגים מסעיף 7.10, רק שהפעם הבאג יושב איפשהו **בהיסטוריה**, לא בשורה שמונחת לפניכם.

### התרחיש: ריפו שקיבלתם, עם באג מוסתר

דמיינו שקיבלתם מעמית למעבדה ריפו עם קוד לחישוב תקופת מטוטלת פשוטה, $T = 2\pi\sqrt{L/g}$, ושלושה קומיטים בהיסטוריה. הקוד רץ בלי שגיאה — אבל כדאי לבדוק את התוצאה לפני שסומכים עליה.

In [ ]:
import tempfile, os

workdir = tempfile.mkdtemp(prefix="git_mystery_")
os.chdir(workdir)

!git init -q -b main
!git config user.email "colleague@example.com"
!git config user.name "Colleague"
!git config color.ui false

In [ ]:
%%writefile pendulum.py
import numpy as np
from scipy import constants

L = 1.0  # מטר
T = 2 * np.pi * np.sqrt(L / constants.g)
print(f"T = {T:.4f} s")

In [ ]:
!git add pendulum.py
!git commit -q -m "חישוב תקופת מטוטלת פשוטה"

In [ ]:
%%writefile pendulum.py
import numpy as np
from scipy import constants

L = 1.0  # מטר
T = 2 * np.pi * np.sqrt(L) / constants.g
print(f"T = {T:.4f} s")

In [ ]:
!git add pendulum.py
!git commit -q -m "ניקוי קוד: פישוט ביטוי החישוב"

In [ ]:
%%writefile pendulum.py
'''חישוב תקופת מטוטלת פשוטה.'''
import numpy as np
from scipy import constants

L = 1.0  # מטר
T = 2 * np.pi * np.sqrt(L) / constants.g
print(f"T = {T:.4f} s")

In [ ]:
!git add pendulum.py
!git commit -q -m "הוספת docstring"

שלושה קומיטים, אף אחד מהם לא נשמע חשוד — "ניקוי קוד" ו"הוספת docstring" נשמעים כמו שינויים תמימים לגמרי. נריץ את הקובץ כפי שהוא עכשיו:

In [ ]:
!python3 pendulum.py

$T \approx 0.64$ שניות עבור מטוטלת של מטר אחד? בדיקת סבירות (שלב 4 מהתהליך שלמדנו בסעיף 7.10) אומרת שזה לא הגיוני — מטוטלת של מטר צריכה תקופה של כשתי שניות. משהו נשבר. בואו נחפש **מתי**, בעזרת ההיסטוריה.

In [ ]:
!git log --oneline

נלכוד את רשימת ה-hashes ישירות ל-Python (עם תחביר הלכידה של IPython, `!פקודה`), כדי שנוכל להשתמש בהם ב-`git diff`/`git show` בלי להעתיק-להדביק ידנית:

In [ ]:
hashes = !git log --format=%H
oldest_first = list(reversed(hashes))
for i, h in enumerate(oldest_first):
    print(i, h)

In [ ]:
# תוכן הקובץ בקומיט הראשון (לפני כל שינוי)
!git show {oldest_first[0]}:pendulum.py

In [ ]:
# מה השתנה בין הקומיט הראשון לשני?
!git diff {oldest_first[0]} {oldest_first[1]} -- pendulum.py

זהו הבאג: `np.sqrt(L) / constants.g` מחשב $\sqrt{L}/g$, לא $\sqrt{L/g}$ — בדיוק הבאג הראשון ממאגר הבאגים של סעיף 7.10, הפעם מוסתר בקומיט עם הודעה מטעה ("ניקוי קוד"). `git diff` בין הקומיט השני לשלישי (בדקו בעצמכם) יראה שהוא לא נגע בשורת החישוב כלל — כל התיקון שצריך הוא לחזור לביטוי של הקומיט הראשון.

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "אחרי git status שמראה שינויים לא-commit-ים, מה ההבדל בין git diff (בלי ארגומנטים) לבין git diff <hash1> <hash2>?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "אין הבדל, שתי הפקודות זהות", "correct": False, "feedback": "לא — הן משוות דברים שונים לגמרי."},
            {"answer": "git diff לבד משווה את הקובץ בתיקיית העבודה מול הקומיט האחרון; git diff עם שני hash-ים משווה בין שני קומיטים בהיסטוריה", "correct": True, "feedback": "נכון."},
            {"answer": "git diff לבד עובד רק על קבצי טקסט, וגם עם hash-ים עובד על כל קובץ", "correct": False, "feedback": "לא קשור — שתי הצורות עובדות על אותם סוגי קבצים."},
            {"answer": "git diff עם hash-ים דורש חיבור לאינטרנט, בלעדיהם לא", "correct": False, "feedback": "כל הפקודות פועלות מקומית בלבד, ללא צורך באינטרנט."}
        ]
    },
    {
        "question": "למה חשוב לבדוק את **סבירות התוצאה הפיזיקלית** (כמו T=0.64s למטוטלת של מטר) כדי לגלות באג כזה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי הקוד זרק שגיאה ברורה שהצביעה בדיוק על השורה הבעייתית", "correct": False, "feedback": "בדיוק ההפך — הקוד רץ בלי שום שגיאה, זה מה שהופך אותו למסוכן."},
            {"answer": "כי אין שום דרך אחרת לגלות באגים מסוג הזה", "correct": False, "feedback": "יש דרכים נוספות (כמו בדיקות אוטומטיות), אבל בדיקת סבירות פיזיקלית היא הכי נגישה וזמינה תמיד."},
            {"answer": "כי באגים של קדימות פעולות וסדר חישוב לא זורקים שגיאה — רק תוצאה שגויה בשקט, ורק היכרות עם סדר הגודל הצפוי חושפת אותם", "correct": True, "feedback": "נכון — זו בדיוק הנקודה שגם 7.10 הדגישה."},
            {"answer": "היא לא חשובה, זו רק תוספת נחמדה", "correct": False, "feedback": "היא קריטית — בלעדיה, כלל לא הייתם יודעים שיש בעיה."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי — בלש קומיטים

קיבלתם ריפו נוסף (נבנה למטה) שמחשב מהירות ממוצעת של כדור, $v = d/t$, על פני שלושה קומיטים. הקוד רץ בלי שגיאה, אבל התוצאה חשודה. השתמשו ב-`git log`, `git diff` ו-`git show` כדי לאתר את הקומיט שהכניס את הבאג ולזהות מה בדיוק הוא — לפני שתציצו בפתרון.

In [ ]:
workdir3 = tempfile.mkdtemp(prefix="git_mystery2_")
os.chdir(workdir3)

!git init -q -b main
!git config user.email "colleague@example.com"
!git config user.name "Colleague"
!git config color.ui false

In [ ]:
%%writefile speed.py
d = 7.0   # מטר
t = 2.0   # שניות
v = d / t
print(f"v = {v} m/s")

In [ ]:
!git add speed.py
!git commit -q -m "חישוב מהירות ממוצעת"

In [ ]:
%%writefile speed.py
d = 7   # מטר
t = 2   # שניות
v = d // t
print(f"v = {v} m/s")

In [ ]:
!git add speed.py
!git commit -q -m "פישוט: הסרת נקודות עשרוניות מיותרות"

In [ ]:
%%writefile speed.py
d = 7   # מטר
t = 2   # שניות
v = d // t
print(f"מהירות ממוצעת: v = {v} m/s")

In [ ]:
!git add speed.py
!git commit -q -m "שיפור הודעת ההדפסה"
!python3 speed.py

In [ ]:
# חקרו כאן: git log, git show, git diff
# hashes3 = !git log --format=%H
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
hashes3 = !git log --format=%H
oldest3 = list(reversed(hashes3))
print(!git show {oldest3[0]}:speed.py)
print(!git diff {oldest3[0]} {oldest3[1]} -- speed.py)
```

הבאג נכנס בקומיט השני ("פישוט: הסרת נקודות עשרוניות מיותרות"): השינוי מ-`d = 7.0` ל-`d = 7` (וכנ"ל ל-`t`) הפך את `d / t` לחילוק בין שני `int`-ים, ואז מישהו (או אותו מפתח, בהמשך) שינה את `/` ל-`//` — חילוק שלמים, שמעגל כלפי מטה. $7/2=3.5$, אך $7 // 2=3$ — התוצאה נראית "סבירה למראה" אבל שגויה, בדיוק כמו באג 2 ממאגר הבאגים של סעיף 7.10. הקומיט השלישי ("שיפור הודעת ההדפסה") לא נגע בחישוב כלל — הוא הסחת דעת תמימה.
`````